# RSNA Knee segmentation-guided ROI validation
Private attributed KneeXNet anatomy-crop comparison. All 58 gold studies remain final evaluation-only, and this notebook cannot submit.


In [ ]:
"""Leakage-safe paired test of physically canonical knee-MRI preprocessing.

Public assets:
* RadImageNet ResNet-50 checkpoint, SHA-256 pinned below.
* Steven Lee Hans' v4 report-derived labels.

All 58 organizer-labelled studies are excluded from training, validation,
early stopping, and model selection. This script cannot create a submission.
"""

from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import resnet50


TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]
PLANES = ("Sagittal", "Coronal", "Axial")
SLOTS = (("Sagittal", True), ("Coronal", True), ("Axial", True),
         ("Sagittal", False), ("Coronal", False), ("Axial", False))
N_SLICE = 3
IMG = 224
CROP_MM = 130.0
TOKEN_DIM = 2048
SEEDS = (20260901, 20260902, 20260903, 20260904, 20260905)
RAD_SHA = "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
FATSAT = re.compile(r"\bfs\b|fat\s*sat|fsat|stir|spair|spir|water\s*excit|fatsup", re.I)
PD = re.compile(r"\bpd\b|\bpdw\b|proton|density|\bdp\b", re.I)
T1 = re.compile(r"\bt1\b|\bt1w\b", re.I)
T2 = re.compile(r"\bt2\b|\bt2w\b", re.I)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
START = time.time()


def log(message):
    print(f"[{time.time() - START:7.1f}s] {message}", flush=True)


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def competition_root():
    for path in (Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                 Path("/kaggle/input/rsna-knee-abnormality-detection")):
        if (path / "train.csv").is_file(): return path
    raise FileNotFoundError("competition data is not attached")


def unique_input(name):
    matches = list(Path("/kaggle/input").rglob(name))
    if len(matches) != 1:
        raise FileNotFoundError(f"expected one {name}, found {matches}")
    return matches[0]


def ffloat(value):
    try: value = float(value)
    except Exception: return None
    return value if math.isfinite(value) else None


def vec(value, n):
    try: value = np.asarray([float(x) for x in value], np.float64)
    except Exception: return None
    return value if len(value) >= n and np.isfinite(value[:n]).all() else None


def classify(ds):
    text = " ".join(str(getattr(ds, key, "") or "") for key in
                    ("SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence"))
    tr, te = ffloat(getattr(ds, "RepetitionTime", None)), ffloat(getattr(ds, "EchoTime", None))
    fs = bool(FATSAT.search(text))
    if T1.search(text) and not T2.search(text) and not PD.search(text): weight = "T1"
    elif T2.search(text) and not PD.search(text): weight = "T2"
    elif PD.search(text): weight = "PD"
    elif tr is not None and tr < 800: weight = "T1"
    elif te is not None and te > 60: weight = "T2"
    elif tr is not None and tr >= 800: weight = "PD"
    else: weight = "UNK"
    return weight, fs


def inspect_series(item):
    study, uid, plane, folder = item
    files = sorted(Path(folder).glob("*.dcm"))
    if not files: return None
    try:
        ds = pydicom.dcmread(files[len(files)//2], stop_before_pixels=True, force=True)
    except Exception: return None
    weight, fs = classify(ds)
    ps = vec(getattr(ds, "PixelSpacing", None), 2)
    iop = vec(getattr(ds, "ImageOrientationPatient", None), 6)
    ipp = vec(getattr(ds, "ImagePositionPatient", None), 3)
    rows, cols = ffloat(getattr(ds, "Rows", None)), ffloat(getattr(ds, "Columns", None))
    centre_x = None
    if ps is not None and iop is not None and ipp is not None and rows and cols:
        centre_x = float((ipp + iop[:3]*ps[1]*cols/2 + iop[3:]*ps[0]*rows/2)[0])
    tag = str(getattr(ds, "ImageLaterality", "") or getattr(ds, "Laterality", "")).upper()
    tag_lat = tag[:1] if tag[:1] in ("L", "R") else None
    geometry_lat = None if centre_x is None or abs(centre_x) < 20 else ("R" if centre_x < 0 else "L")
    # Geometry is canonical when decisive; the DICOM tag is only a fallback.
    lat = geometry_lat or tag_lat
    return {"study": study, "uid": uid, "plane": plane, "folder": str(folder),
            "files": [str(x) for x in files], "n": len(files), "weight": weight,
            "fs": fs, "ps": ps.tolist() if ps is not None else None, "lat": lat}


def choose(records, plane, fluid, canonical):
    candidates = [r for r in records if r["plane"] == plane]
    if canonical:
        if fluid:
            ranked = [(0 if r["weight"] == "PD" and r["fs"] else
                       1 if r["weight"] == "T2" and r["fs"] else
                       2 if r["fs"] else 9, -r["n"], r) for r in candidates]
        else:
            ranked = [(0 if r["weight"] == "PD" and not r["fs"] else
                       1 if r["weight"] == "T1" and not r["fs"] else
                       2 if r["weight"] == "T2" and not r["fs"] else 9,
                       -r["n"], r) for r in candidates]
        ranked = [x for x in ranked if x[0] < 9]
        return sorted(ranked, key=lambda x: (x[0], x[1]))[0][2] if ranked else None
    preferred = [r for r in candidates if bool(r["fs"]) == bool(fluid)]
    pool = preferred or candidates
    return max(pool, key=lambda r: r["n"]) if pool else None


def ordered_headers(record):
    rows = []
    for i, path in enumerate(record["files"]):
        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True, force=True,
                                 specific_tags=["ImageOrientationPatient", "ImagePositionPatient", "InstanceNumber"])
            iop, ipp = vec(getattr(ds, "ImageOrientationPatient", None), 6), vec(getattr(ds, "ImagePositionPatient", None), 3)
            projection = float(np.dot(ipp, np.cross(iop[:3], iop[3:]))) if iop is not None and ipp is not None else None
            instance = ffloat(getattr(ds, "InstanceNumber", None))
        except Exception: projection, instance = None, None
        rows.append((path, projection, instance, i))
    if all(r[1] is not None for r in rows): rows.sort(key=lambda r: r[1])
    elif all(r[2] is not None for r in rows): rows.sort(key=lambda r: r[2])
    return rows


def selected_rows(record, canonical):
    rows = ordered_headers(record)
    if not rows: return []
    if not canonical:
        indices = np.linspace(.20*(len(rows)-1), .80*(len(rows)-1), N_SLICE).round().astype(int)
        return [rows[i] for i in indices]
    positions = np.asarray([r[1] if r[1] is not None else np.nan for r in rows])
    if np.isfinite(positions).all() and len(rows) > 1:
        centre = float(np.median(positions)); span = 35.0 if record["plane"] != "Axial" else 24.0
        targets = np.linspace(centre-span, centre+span, N_SLICE)
        return [rows[int(np.argmin(abs(positions-target)))] for target in targets]
    indices = np.linspace(.10*(len(rows)-1), .90*(len(rows)-1), N_SLICE).round().astype(int)
    return [rows[i] for i in indices]


def read_pixels(path):
    ds = pydicom.dcmread(path, force=True)
    image = ds.pixel_array.astype(np.float32)
    image = image * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1": image = image.max() - image
    ps = vec(getattr(ds, "PixelSpacing", None), 2)
    return image, ps


def render_slot(record, canonical):
    output = np.zeros((N_SLICE, IMG, IMG), np.uint8)
    if record is None: return output, 0
    arrays = []
    for row in selected_rows(record, canonical):
        try: arrays.append(read_pixels(row[0]))
        except Exception: arrays.append((None, None))
    valid = [a for a, _ in arrays if a is not None]
    if not valid: return output, 0
    values = np.concatenate([a.ravel()[::max(1, a.size//20000)] for a in valid])
    if canonical:
        centre = float(np.median(values)); scale = 1.4826*float(np.median(abs(values-centre)))
        if scale < 1e-6: scale = max(float(np.std(values)), 1e-6)
        low, high = centre-4*scale, centre+4*scale
    else:
        low, high = np.percentile(values, [1, 99])
    for i, (image, ps) in enumerate(arrays):
        if image is None: continue
        h, w = image.shape
        if ps is None: ps = np.asarray((.5, .5))
        if canonical:
            crop_h = min(h, max(16, int(round(CROP_MM/max(ps[0], 1e-3)))))
            crop_w = min(w, max(16, int(round(CROP_MM/max(ps[1], 1e-3)))))
        else:
            crop_h = crop_w = min(h, w, max(16, int(round(CROP_MM/max(ps[0], 1e-3)))))
        y0, x0 = (h-crop_h)//2, (w-crop_w)//2
        image = image[y0:y0+crop_h, x0:x0+crop_w]
        image = np.clip((image-low)/max(high-low, 1e-6), 0, 1)
        image = cv2.resize(image, (IMG, IMG), interpolation=cv2.INTER_AREA)
        if canonical and record.get("lat") == "R" and record["plane"] in ("Coronal", "Axial"):
            image = image[:, ::-1]
        output[i] = np.round(image*255).astype(np.uint8)
    return output, 1


class Encoder(nn.Module):
    def __init__(self):
        super().__init__(); self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])
    def forward(self, x): return self.backbone(x).mean((2, 3))


def load_encoder():
    path = unique_input("ResNet50.pt")
    if sha256(path) != RAD_SHA: raise RuntimeError("RadImageNet checkpoint hash drift")
    model = Encoder(); state = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True); model.eval().to(DEVICE)
    for p in model.parameters(): p.requires_grad_(False)
    return model


def encode_arm(studies, by_study, canonical):
    features = np.zeros((len(studies), len(SLOTS)*N_SLICE, TOKEN_DIM), np.float16)
    masks = np.zeros((len(studies), len(SLOTS)*N_SLICE), np.float32)
    encoder = load_encoder(); batch_study = 24
    for start in range(0, len(studies), batch_study):
        ids = studies[start:start+batch_study]
        jobs = []
        for uid in ids:
            records = by_study.get(uid, [])
            jobs.extend((choose(records, plane, fluid, canonical), canonical) for plane, fluid in SLOTS)
        with ThreadPoolExecutor(max_workers=12) as pool: rendered = list(pool.map(lambda x: render_slot(*x), jobs))
        pixels = np.stack([x[0] for x in rendered]).reshape(len(ids), len(SLOTS)*N_SLICE, IMG, IMG)
        slot_ok = np.asarray([x[1] for x in rendered], np.float32).reshape(len(ids), len(SLOTS))
        token_ok = np.repeat(slot_ok, N_SLICE, axis=1)
        flat = pixels.reshape(-1, IMG, IMG); valid = np.flatnonzero(token_ok.reshape(-1)>0)
        encoded = np.zeros((len(flat), TOKEN_DIM), np.float16)
        for b0 in range(0, len(valid), 128):
            ix = valid[b0:b0+128]
            x = torch.from_numpy(flat[ix]).to(DEVICE).float().div_(127.5).sub_(1).unsqueeze(1).expand(-1,3,-1,-1)
            with torch.autocast("cuda", enabled=DEVICE.type=="cuda"):
                y = encoder(x).float().cpu().numpy().astype(np.float16)
            encoded[ix] = y
        features[start:start+len(ids)] = encoded.reshape(len(ids), -1, TOKEN_DIM)
        masks[start:start+len(ids)] = token_ok
        if start % (batch_study*10) == 0: log(f"{'canonical' if canonical else 'incumbent'} encoded {start}/{len(studies)}")
    del encoder; gc.collect(); torch.cuda.empty_cache()
    return features, masks


class AttentionHead(nn.Module):
    def __init__(self):
        super().__init__(); self.norm=nn.LayerNorm(TOKEN_DIM); self.proj=nn.Linear(TOKEN_DIM,256)
        self.att=nn.Linear(256,len(TARGETS)); self.cls=nn.Parameter(torch.empty(len(TARGETS),256)); self.bias=nn.Parameter(torch.zeros(len(TARGETS)))
        nn.init.trunc_normal_(self.cls,std=.02)
    def forward(self, x, mask):
        h=torch.nn.functional.gelu(self.proj(self.norm(x.float()))); a=self.att(h)
        a=a.masked_fill(mask[:,:,None]<.5,-1e4); a=torch.softmax(a,dim=1)
        pooled=torch.einsum("btn,btf->bnf",a,h)
        return (pooled*self.cls).sum(-1)+self.bias


def macro_auc(y, p):
    scores=[]
    for j in range(y.shape[1]):
        truth = y[:, j]
        if not np.isin(np.unique(truth), (0.0, 1.0)).all():
            truth = (truth >= 0.5).astype(np.uint8)
        if len(np.unique(truth)) > 1:
            scores.append(roc_auc_score(truth, p[:, j]))
    return float(np.mean(scores))


def train_head(features, masks, labels, train_idx, val_idx, seed):
    seed_all(seed); model=AttentionHead().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-3)
    tx=torch.from_numpy(features[train_idx]); tm=torch.from_numpy(masks[train_idx]); ty=torch.from_numpy(labels[train_idx])
    loader=DataLoader(TensorDataset(tx,tm,ty),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(seed))
    best, best_score = None, -1
    for epoch in range(20):
        model.train()
        for x,m,y in loader:
            x,m,y=x.to(DEVICE),m.to(DEVICE),y.to(DEVICE); opt.zero_grad(set_to_none=True)
            loss=torch.nn.functional.binary_cross_entropy_with_logits(model(x,m),y.float()); loss.backward(); opt.step()
        pred=predict(model,features,masks,val_idx); score=macro_auc(labels[val_idx],pred)
        if score>best_score: best_score=score; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best); return predict(model,features,masks,np.arange(len(features))), best_score


@torch.no_grad()
def predict(model, features, masks, indices):
    model.eval(); out=[]
    for b0 in range(0,len(indices),128):
        ix=indices[b0:b0+128]; x=torch.from_numpy(features[ix]).to(DEVICE); m=torch.from_numpy(masks[ix]).to(DEVICE)
        out.append(torch.sigmoid(model(x,m)).cpu().numpy())
    return np.concatenate(out)


def paired_bootstrap(y, inc, can, draws=5000, seed=20260901):
    rng=np.random.default_rng(seed); delta=[]
    for _ in range(draws):
        ix=rng.integers(0,len(y),len(y)); delta.append(macro_auc(y[ix],can[ix])-macro_auc(y[ix],inc[ix]))
    d=np.asarray(delta); return {"mean":float(d.mean()),"p_improve":float((d>0).mean()),"q025":float(np.quantile(d,.025)),"q975":float(np.quantile(d,.975))}


def main():
    if DEVICE.type!="cuda": raise RuntimeError("paired validation requires GPU")
    root=competition_root(); train=pd.read_csv(root/"train.csv",dtype={"StudyInstanceUID":str})
    gold=train[TARGETS].notna().all(axis=1).to_numpy();
    if int(gold.sum())!=58: raise RuntimeError(f"expected 58 gold studies, found {gold.sum()}")
    label_path=unique_input("llm_labels_v4_blend.csv"); weak=pd.read_csv(label_path,dtype={"StudyInstanceUID":str})
    aligned=train[["StudyInstanceUID"]].merge(weak,on="StudyInstanceUID",how="left",validate="one_to_one")
    labels=aligned[TARGETS].to_numpy(np.float32)
    if not np.isfinite(labels).all(): raise RuntimeError("weak labels incomplete")
    series=pd.read_csv(root/"train_series.csv",dtype=str); series_root=root/("train_series" if (root/"train_series").is_dir() else "train_images")
    items=[(r.StudyInstanceUID,r.SeriesInstanceUID,r.Anatomical_Plane,series_root/r.StudyInstanceUID/r.SeriesInstanceUID) for r in series.itertuples(index=False)]
    with ThreadPoolExecutor(max_workers=16) as pool: records=[r for r in pool.map(inspect_series,items) if r]
    by_study={}
    for record in records: by_study.setdefault(record["study"],[]).append(record)
    studies=train.StudyInstanceUID.tolist(); inc_f,inc_m=encode_arm(studies,by_study,False); can_f,can_m=encode_arm(studies,by_study,True)
    nongold=np.flatnonzero(~gold); gold_idx=np.flatnonzero(gold); results=[]; inc_gold=[]; can_gold=[]
    for seed in SEEDS:
        rng=np.random.default_rng(seed); perm=rng.permutation(nongold); cut=int(.9*len(perm)); tr,va=perm[:cut],perm[cut:]
        inc_pred,inc_weak=train_head(inc_f,inc_m,labels,tr,va,seed); can_pred,can_weak=train_head(can_f,can_m,labels,tr,va,seed)
        ia=macro_auc(train.loc[gold,TARGETS].to_numpy(float),inc_pred[gold_idx]); ca=macro_auc(train.loc[gold,TARGETS].to_numpy(float),can_pred[gold_idx])
        results.append({"seed":seed,"incumbent_weak_val_auc":inc_weak,"canonical_weak_val_auc":can_weak,"incumbent_gold_auc":ia,"canonical_gold_auc":ca,"delta":ca-ia})
        inc_gold.append(inc_pred[gold_idx]); can_gold.append(can_pred[gold_idx]); log(f"seed {seed}: gold delta {ca-ia:+.6f}")
    y_gold=train.loc[gold,TARGETS].to_numpy(float); inc_mean=np.mean(inc_gold,axis=0); can_mean=np.mean(can_gold,axis=0)
    boot=paired_bootstrap(y_gold,inc_mean,can_mean); deltas=np.asarray([x["delta"] for x in results])
    gate={"delta_at_least_0_005":float(macro_auc(y_gold,can_mean)-macro_auc(y_gold,inc_mean))>=.005,
          "positive_seeds_at_least_4":int((deltas>0).sum())>=4,"bootstrap_p_at_least_0_95":boot["p_improve"]>=.95,"bootstrap_q025_positive":boot["q025"]>0}
    report={"contract":"private paired validation; gold evaluation only; no submission","encoder":"RadImageNet ResNet-50","encoder_sha256":RAD_SHA,
            "weak_labels":"stevenleehans llm_labels_v4_blend.csv","gold_fit_count":0,"gold_evaluation_count":58,"seeds":results,
            "ensemble":{"incumbent_gold_auc":macro_auc(y_gold,inc_mean),"canonical_gold_auc":macro_auc(y_gold,can_mean),"delta":macro_auc(y_gold,can_mean)-macro_auc(y_gold,inc_mean)},
            "bootstrap":boot,"gate":gate,"passed":bool(all(gate.values())),"submission_created":False}
    Path("/kaggle/working/physical_paired_validation_report.json").write_text(json.dumps(report,indent=2)+"\n")
    print(json.dumps(report,indent=2))








"""Private KneeXNet-guided anatomy ROI validation.

The candidate uses an attributed knee cartilage/meniscus segmentation model to
replace only confident sagittal RadImageNet tokens with anatomy-focused crops.
The fixed hybrid uses those predictions only for meniscus and OA targets.
All 58 organizer-labelled studies remain final evaluation-only.
"""

import subprocess
import sys
import urllib.request

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "segmentation-models-pytorch==0.5.0",
])
import segmentation_models_pytorch as smp


SEGMENTATION_LFS_BATCH_URL = (
    "https://github.com/pitthexai/"
    "Knee_MRI_Segmentation_2.5D.git/info/lfs/objects/batch"
)
SEGMENTATION_SHA256 = "0c2b5fe87f86e78c7d90e9dfc3de1307aaef57c36d44ebb1251eecdcffc3c663"
SEGMENTATION_SIZE = 97924670
SEGMENTATION_COMMIT = "f2452feb602f2bd9e790550c8a5a07f62e7d6e55"
SEGMENTATION_TARGETS = (2, 3, 4, 5, 6)
SAGITTAL_SLOT_INDICES = (0, 3)
SEG_INPUT = 256


def load_segmentation_model():
    path = Path("/kaggle/working/kneexnet_baseline_best_model.pth")
    if not path.is_file():
        body = json.dumps({
            "operation": "download",
            "transfers": ["basic"],
            "objects": [{"oid": SEGMENTATION_SHA256, "size": SEGMENTATION_SIZE}],
        }).encode("utf-8")
        request = urllib.request.Request(
            SEGMENTATION_LFS_BATCH_URL,
            data=body,
            headers={
                "Accept": "application/vnd.git-lfs+json",
                "Content-Type": "application/vnd.git-lfs+json",
            },
            method="POST",
        )
        with urllib.request.urlopen(request, timeout=60) as response:
            metadata = json.loads(response.read().decode("utf-8"))
        download_url = metadata["objects"][0]["actions"]["download"]["href"]
        urllib.request.urlretrieve(download_url, path)
    if path.stat().st_size != SEGMENTATION_SIZE:
        raise RuntimeError("KneeXNet checkpoint size drift")
    if sha256(path) != SEGMENTATION_SHA256:
        raise RuntimeError("KneeXNet checkpoint hash drift")
    model = smp.Unet(
        encoder_name="resnet34", encoder_weights=None,
        in_channels=3, classes=5,
    )
    state = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    model.eval().to(DEVICE)
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model


def load_frozen_radimagenet():
    model = load_encoder()
    model.eval()
    return model


def normalized_triplets(record):
    rows = ordered_headers(record)
    if not rows:
        return []
    centres = np.linspace(0.20 * (len(rows) - 1), 0.80 * (len(rows) - 1), N_SLICE).round().astype(int)
    output = []
    for centre in centres:
        indices = [max(0, centre - 1), centre, min(len(rows) - 1, centre + 1)]
        arrays = []
        try:
            for index in indices:
                image, _ = read_pixels(rows[index][0])
                height, width = image.shape
                side = min(height, width)
                y0, x0 = (height - side) // 2, (width - side) // 2
                arrays.append(image[y0:y0 + side, x0:x0 + side])
        except Exception:
            continue
        values = np.concatenate([image.ravel()[::max(1, image.size // 20000)] for image in arrays])
        low, high = np.percentile(values, (1, 99))
        stack = []
        for image in arrays:
            image = np.clip((image - low) / max(high - low, 1e-6), 0, 1)
            image = cv2.resize(image, (SEG_INPUT, SEG_INPUT), interpolation=cv2.INTER_AREA)
            stack.append(image.astype(np.float32))
        output.append(np.stack(stack))
    return output


def confident_roi(probability, centre_image):
    foreground = probability[1:].max(axis=0)
    mask = foreground >= 0.50
    area = float(mask.mean())
    rows, columns = np.where(mask)
    if len(rows) < 32 or area < 0.005 or area > 0.60:
        return None, area
    y0, y1 = int(rows.min()), int(rows.max()) + 1
    x0, x1 = int(columns.min()), int(columns.max()) + 1
    height, width = y1 - y0, x1 - x0
    if min(height, width) < 12:
        return None, area
    margin_y, margin_x = max(8, int(0.25 * height)), max(8, int(0.25 * width))
    y0, y1 = max(0, y0 - margin_y), min(SEG_INPUT, y1 + margin_y)
    x0, x1 = max(0, x0 - margin_x), min(SEG_INPUT, x1 + margin_x)
    crop = centre_image[y0:y1, x0:x1]
    if min(crop.shape) < 16:
        return None, area
    crop = cv2.resize(crop, (IMG, IMG), interpolation=cv2.INTER_AREA)
    return np.round(np.clip(crop, 0, 1) * 255).astype(np.uint8), area


@torch.no_grad()
def build_segmentation_roi_features(studies, by_study, control_features, control_masks):
    candidate = np.array(control_features, copy=True)
    segmentation = load_segmentation_model()
    encoder = load_frozen_radimagenet()
    audit = {"attempted": 0, "confident": 0, "fallback": 0, "foreground_areas": []}
    batch_studies = 32
    for start in range(0, len(studies), batch_studies):
        ids = studies[start:start + batch_studies]
        stacks, assignments = [], []
        for local_index, uid in enumerate(ids):
            records = by_study.get(uid, [])
            for slot_index in SAGITTAL_SLOT_INDICES:
                plane, fluid = SLOTS[slot_index]
                record = choose(records, plane, fluid, False)
                if record is None:
                    continue
                for slice_index, stack in enumerate(normalized_triplets(record)):
                    stacks.append(stack)
                    assignments.append((local_index, slot_index, slice_index))
        if stacks:
            probabilities = []
            array = np.stack(stacks).astype(np.float32)
            for offset in range(0, len(array), 64):
                tensor = torch.from_numpy(array[offset:offset + 64]).to(DEVICE)
                with torch.autocast("cuda", enabled=DEVICE.type == "cuda"):
                    probability = torch.softmax(segmentation(tensor), dim=1)
                probabilities.append(probability.float().cpu().numpy())
            probabilities = np.concatenate(probabilities)
            roi_images, roi_assignments = [], []
            for probability, stack, assignment in zip(probabilities, stacks, assignments):
                audit["attempted"] += 1
                roi, area = confident_roi(probability, stack[1])
                audit["foreground_areas"].append(area)
                if roi is None:
                    audit["fallback"] += 1
                    continue
                audit["confident"] += 1
                roi_images.append(roi)
                roi_assignments.append(assignment)
            if roi_images:
                encoded = []
                array = np.stack(roi_images)
                for offset in range(0, len(array), 128):
                    tensor = torch.from_numpy(array[offset:offset + 128]).to(DEVICE).float()
                    tensor = tensor.div_(127.5).sub_(1).unsqueeze(1).expand(-1, 3, -1, -1)
                    with torch.autocast("cuda", enabled=DEVICE.type == "cuda"):
                        encoded.append(encoder(tensor).float().cpu().numpy().astype(np.float16))
                encoded = np.concatenate(encoded)
                for feature, (local_index, slot_index, slice_index) in zip(encoded, roi_assignments):
                    study_index = start + local_index
                    token_index = slot_index * N_SLICE + slice_index
                    if control_masks[study_index, token_index] > 0:
                        candidate[study_index, token_index] = feature
        if start % (batch_studies * 10) == 0:
            log(f"segmentation ROI encoded {start}/{len(studies)}")
    areas = np.asarray(audit.pop("foreground_areas"), dtype=float)
    audit.update({
        "confidence_rate": audit["confident"] / max(1, audit["attempted"]),
        "foreground_area_median": float(np.median(areas)) if len(areas) else None,
        "foreground_area_q05": float(np.quantile(areas, 0.05)) if len(areas) else None,
        "foreground_area_q95": float(np.quantile(areas, 0.95)) if len(areas) else None,
    })
    del segmentation, encoder
    gc.collect(); torch.cuda.empty_cache()
    return candidate, audit


def normalized_reports_roi(series):
    return (series.fillna("").astype(str).str.lower()
            .str.replace(r"\s+", " ", regex=True).str.strip())


def report_groups_roi(series):
    return np.asarray([hashlib.sha256(text.encode()).hexdigest()[:24]
                       for text in normalized_reports_roi(series)])


def segmentation_roi_main():
    if DEVICE.type != "cuda":
        raise RuntimeError("segmentation ROI validation requires GPU")
    root = competition_root()
    train = pd.read_csv(root / "train.csv", dtype={"StudyInstanceUID": str})
    studies = train.StudyInstanceUID.tolist()
    gold = train[TARGETS].notna().all(axis=1).to_numpy()
    if int(gold.sum()) != 58:
        raise RuntimeError(f"expected 58 gold studies, found {gold.sum()}")
    gold_index = np.flatnonzero(gold)
    non_gold = np.flatnonzero(~gold)

    weak = pd.read_csv(unique_input("llm_labels_v4_blend.csv"), dtype={"StudyInstanceUID": str})
    aligned = train[["StudyInstanceUID"]].merge(weak, on="StudyInstanceUID", how="left", validate="one_to_one")
    labels = aligned[TARGETS].to_numpy(np.float32)
    if not np.isfinite(labels).all():
        raise RuntimeError("weak labels incomplete")

    package = np.load(unique_input("incumbent_radimagenet_features.npz"), allow_pickle=False)
    if package["ids"].astype(str).tolist() != studies:
        raise RuntimeError("incumbent feature identity/order drift")
    if str(package["encoder_sha256"]) != RAD_SHA:
        raise RuntimeError("incumbent encoder hash drift")
    control_features, control_masks = package["features"], package["masks"]

    series = pd.read_csv(root / "train_series.csv", dtype=str)
    series_root = root / ("train_series" if (root / "train_series").is_dir() else "train_images")
    items = [(row.StudyInstanceUID, row.SeriesInstanceUID, row.Anatomical_Plane,
              series_root / row.StudyInstanceUID / row.SeriesInstanceUID)
             for row in series.itertuples(index=False)]
    with ThreadPoolExecutor(max_workers=16) as pool:
        records = [record for record in pool.map(inspect_series, items) if record]
    by_study = {}
    for record in records:
        by_study.setdefault(record["study"], []).append(record)
    candidate_features, roi_audit = build_segmentation_roi_features(
        studies, by_study, control_features, control_masks
    )
    if roi_audit["confidence_rate"] < 0.25:
        raise RuntimeError(f"segmentation confidence coverage too low: {roi_audit}")

    groups = report_groups_roi(train["Report"])
    y_gold = train.loc[gold, TARGETS].to_numpy(float)
    results, control_gold, hybrid_gold = [], [], []
    for seed in SEEDS:
        unique_groups = np.unique(groups[non_gold])
        shuffled = np.random.default_rng(seed).permutation(unique_groups)
        validation_groups = set(shuffled[:max(1, int(0.10 * len(shuffled)))])
        validation_index = non_gold[np.asarray([group in validation_groups for group in groups[non_gold]])]
        train_index = non_gold[np.asarray([group not in validation_groups for group in groups[non_gold]])]
        if set(groups[train_index]).intersection(groups[validation_index]):
            raise RuntimeError("report-group leakage")
        control_prediction, control_weak = train_head(
            control_features, control_masks, labels, train_index, validation_index, seed
        )
        roi_prediction, roi_weak = train_head(
            candidate_features, control_masks, labels, train_index, validation_index, seed
        )
        hybrid_prediction = np.array(control_prediction, copy=True)
        hybrid_prediction[:, SEGMENTATION_TARGETS] = roi_prediction[:, SEGMENTATION_TARGETS]
        hybrid_weak = macro_auc(labels[validation_index], hybrid_prediction[validation_index])
        control_auc = macro_auc(y_gold, control_prediction[gold_index])
        hybrid_auc = macro_auc(y_gold, hybrid_prediction[gold_index])
        results.append({
            "seed": seed,
            "control_weak_validation_auc": control_weak,
            "roi_head_weak_validation_auc": roi_weak,
            "hybrid_weak_validation_auc": hybrid_weak,
            "control_gold_auc": control_auc,
            "hybrid_gold_auc": hybrid_auc,
            "delta": hybrid_auc - control_auc,
        })
        control_gold.append(control_prediction[gold_index])
        hybrid_gold.append(hybrid_prediction[gold_index])
        log(f"seed {seed}: segmentation-ROI hybrid delta {hybrid_auc - control_auc:+.6f}")

    control_mean = np.mean(control_gold, axis=0)
    hybrid_mean = np.mean(hybrid_gold, axis=0)
    control_auc = macro_auc(y_gold, control_mean)
    hybrid_auc = macro_auc(y_gold, hybrid_mean)
    bootstrap = paired_bootstrap(y_gold, control_mean, hybrid_mean)
    deltas = np.asarray([result["delta"] for result in results])
    per_target_delta = {
        TARGETS[index]: float(roc_auc_score(y_gold[:, index], hybrid_mean[:, index]) -
                              roc_auc_score(y_gold[:, index], control_mean[:, index]))
        for index in SEGMENTATION_TARGETS
    }
    gate = {
        "delta_at_least_0_005": hybrid_auc - control_auc >= 0.005,
        "positive_seeds_at_least_4": int((deltas > 0).sum()) >= 4,
        "bootstrap_p_at_least_0_95": bootstrap["p_improve"] >= 0.95,
        "bootstrap_q025_positive": bootstrap["q025"] > 0,
    }
    report = {
        "contract": "private attributed segmentation-guided ROI validation; gold evaluation only; no submission",
        "encoder": "RadImageNet ResNet-50",
        "encoder_sha256": RAD_SHA,
        "segmentation_model": "KneeXNet-2.5D baseline ResNet34 U-Net",
        "segmentation_repository": "pitthexai/Knee_MRI_Segmentation_2.5D",
        "segmentation_commit": SEGMENTATION_COMMIT,
        "segmentation_checkpoint_sha256": SEGMENTATION_SHA256,
        "segmentation_citation": "Sanogo et al., KneeXNet-2.5D, npj Health Systems (2026), doi:10.1038/s44401-026-00072-5",
        "segmentation_use": "research/educational private validation",
        "hybrid_targets": [TARGETS[index] for index in SEGMENTATION_TARGETS],
        "roi_audit": roi_audit,
        "weak_labels": "stevenleehans llm_labels_v4_blend.csv",
        "gold_fit_count": 0,
        "gold_evaluation_count": 58,
        "seeds": results,
        "ensemble": {
            "control_gold_auc": control_auc,
            "hybrid_gold_auc": hybrid_auc,
            "delta": hybrid_auc - control_auc,
            "per_target_delta": per_target_delta,
        },
        "bootstrap": bootstrap,
        "gate": gate,
        "passed": bool(all(gate.values())),
        "submission_created": False,
    }
    Path("/kaggle/working/segmentation_roi_validation_report.json").write_text(
        json.dumps(report, indent=2) + "\n", encoding="utf-8"
    )
    print(json.dumps(report, indent=2))


segmentation_roi_main()
